# Verify v5 vs v6 — Two-stage Real Audio Comparison

**วัตถุประสงค์:** เปรียบเทียบ Two-stage v5 (ไม่ clean) vs v6 (clean) บน real-world audio (TikTok)

**คำถามวิจัย:** v5 F1 = 0.887 vs v6 F1 = 0.82 — ต่างกันจริงไหม หรือเป็น statistical noise?

**Method:**
1. โหลด predictions per window
2. Aggregate per file (strategy='any')
3. Confusion matrix + classification report
4. List ไฟล์ที่ predict ต่างกัน
5. Bootstrap confidence interval
6. (Optional) ฟังเสียงไฟล์ที่ต่าง

**ผลลัพธ์ที่คาดได้:**
- ต่างกันแค่ 1-2 ไฟล์ใน 19
- 95% CI overlap → not statistically significant
- ไม่สามารถสรุปว่า v5 ดีกว่า v6 ได้จาก data นี้


In [ ]:
# === Cell 1: Setup ===
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                              f1_score, accuracy_score)

DRIVE_BASE = '/content/drive/MyDrive/ThaiScamCall'
RUNS_DIR   = f'{DRIVE_BASE}/runs/experiment2'   # 🔧 ปรับ path ถ้าต่าง
REAL_AUDIO = f'{DRIVE_BASE}/real_audio'

CLASS_NAMES = ['not_scam', 'scam']
print(f'Runs dir: {RUNS_DIR}')


In [ ]:
# === Cell 2: Load predictions (ปรับ filename ตามจริง) ===
# 🔧 ปรับ filename ตามที่อยู่จริงใน RUNS_DIR
V5_FILE = 'two_stage_streaming_rounds_v5.csv'   # หรือชื่อจริง
V6_FILE = 'two_stage_streaming_rounds_v6.csv'   # หรือชื่อจริง

# โหลด
print('Files ใน RUNS_DIR:')
for f in sorted(os.listdir(RUNS_DIR)):
    print(f'  {f}')

# ลองโหลด — ถ้าไม่เจอแก้ชื่อข้างบน
def load_pred(fn):
    p = f'{RUNS_DIR}/{fn}'
    return pd.read_csv(p) if os.path.exists(p) else None

df_v5 = load_pred(V5_FILE)
df_v6 = load_pred(V6_FILE)
print(f'\nv5 shape: {df_v5.shape if df_v5 is not None else "❌ NOT FOUND"}')
print(f'v6 shape: {df_v6.shape if df_v6 is not None else "❌ NOT FOUND"}')


In [ ]:
# === Cell 3: Aggregate per file (any strategy) ===
def aggregate_any(df):
    """1 prediction per file: scam ถ้ามี window ไหน trigger"""
    calls = []
    for fn, g in df.groupby('filename'):
        calls.append({
            'filename': fn,
            'true':     int(g.true_label.iloc[0]),
            'pred_any': int((g.prob_scam > 0.5).any()),
            'max_prob': float(g.prob_scam.max()),
            'mean_prob': float(g.prob_scam.mean()),
            'n_rounds': len(g),
        })
    return pd.DataFrame(calls)

agg_v5 = aggregate_any(df_v5) if df_v5 is not None else None
agg_v6 = aggregate_any(df_v6) if df_v6 is not None else None

if agg_v5 is not None:
    print(f'v5 files: {len(agg_v5)}')
    print(f'v6 files: {len(agg_v6)}')
    print(f'\nClass distribution (v5):')
    print(agg_v5.true.value_counts().rename({0:"not_scam", 1:"scam"}))


In [ ]:
# === Cell 4: Confusion matrix + classification report ===
def report(agg, name):
    y, p = agg.true.values, agg.pred_any.values
    cm = confusion_matrix(y, p)
    f1 = f1_score(y, p, average='macro')
    acc = accuracy_score(y, p)
    print(f'\n========== {name} ==========')
    print(f'Macro F1: {f1:.4f}')
    print(f'Accuracy: {acc:.4f}')
    print(f'Errors:   {(y != p).sum()}/{len(y)}')
    print(f'\nConfusion matrix:')
    print(f'  TN={cm[0,0]:>2}  FP={cm[0,1]:>2}')
    print(f'  FN={cm[1,0]:>2}  TP={cm[1,1]:>2}')
    print(f'\n{classification_report(y, p, target_names=CLASS_NAMES, digits=4)}')
    return f1, cm

if agg_v5 is not None:
    f1_v5, cm_v5 = report(agg_v5, 'v5 (ไม่ clean)')
if agg_v6 is not None:
    f1_v6, cm_v6 = report(agg_v6, 'v6 (clean)')

# Plot side-by-side
if agg_v5 is not None and agg_v6 is not None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, cm, name, f1 in [(axes[0], cm_v5, 'v5 (ไม่ clean)', f1_v5),
                              (axes[1], cm_v6, 'v6 (clean)', f1_v6)]:
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        ax.set_title(f'{name}\nMacro F1 = {f1:.4f}')
    plt.tight_layout()
    plt.show()


In [ ]:
# === Cell 5: ไฟล์ที่ predict ต่างกัน ===
if agg_v5 is not None and agg_v6 is not None:
    # join
    merged = agg_v5[['filename','true','pred_any','max_prob']].rename(
        columns={'pred_any':'pred_v5', 'max_prob':'maxp_v5'})
    merged = merged.merge(
        agg_v6[['filename','pred_any','max_prob']].rename(
            columns={'pred_any':'pred_v6', 'max_prob':'maxp_v6'}),
        on='filename'
    )

    diff = merged[merged.pred_v5 != merged.pred_v6].copy()
    print(f'\n=== ไฟล์ที่ v5 vs v6 predict ต่างกัน: {len(diff)} ไฟล์ ===\n')

    if len(diff) > 0:
        diff['v5_correct'] = (diff.pred_v5 == diff.true).map({True: '✅', False: '❌'})
        diff['v6_correct'] = (diff.pred_v6 == diff.true).map({True: '✅', False: '❌'})
        cols = ['filename', 'true', 'pred_v5', 'v5_correct', 'maxp_v5',
                'pred_v6', 'v6_correct', 'maxp_v6']
        print(diff[cols].to_string(index=False))
    else:
        print('ทุกไฟล์ predict เหมือนกัน — model ผลเหมือนกัน')

    # Count
    v5_only_correct = ((merged.pred_v5 == merged.true) & (merged.pred_v6 != merged.true)).sum()
    v6_only_correct = ((merged.pred_v6 == merged.true) & (merged.pred_v5 != merged.true)).sum()
    print(f'\n📊 สรุป:')
    print(f'   v5 ถูก แต่ v6 ผิด:  {v5_only_correct} ไฟล์')
    print(f'   v6 ถูก แต่ v5 ผิด:  {v6_only_correct} ไฟล์')
    print(f'   ทั้งคู่ถูก:          {((merged.pred_v5 == merged.true) & (merged.pred_v6 == merged.true)).sum()}')
    print(f'   ทั้งคู่ผิด:          {((merged.pred_v5 != merged.true) & (merged.pred_v6 != merged.true)).sum()}')


In [ ]:
# === Cell 6: Bootstrap confidence interval (statistical significance) ===
# Bootstrap F1 จากการ resample เพื่อดู uncertainty
def bootstrap_f1(agg, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed)
    y, p = agg.true.values, agg.pred_any.values
    n = len(y)
    f1s = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        try:
            f1s.append(f1_score(y[idx], p[idx], average='macro', zero_division=0))
        except:
            pass
    return np.array(f1s)

if agg_v5 is not None and agg_v6 is not None:
    boot_v5 = bootstrap_f1(agg_v5)
    boot_v6 = bootstrap_f1(agg_v6)

    print(f'\n=== Bootstrap F1 (n=10000) ===')
    print(f'v5: mean={boot_v5.mean():.4f}, 95% CI = [{np.percentile(boot_v5, 2.5):.4f}, {np.percentile(boot_v5, 97.5):.4f}]')
    print(f'v6: mean={boot_v6.mean():.4f}, 95% CI = [{np.percentile(boot_v6, 2.5):.4f}, {np.percentile(boot_v6, 97.5):.4f}]')

    # Probability v5 > v6
    diff = boot_v5 - boot_v6
    p_v5_better = (diff > 0).mean()
    print(f'\n📊 P(v5 F1 > v6 F1) = {p_v5_better:.3f}')
    print(f'    F1 difference (v5 - v6): mean={diff.mean():.4f}, 95% CI = [{np.percentile(diff, 2.5):.4f}, {np.percentile(diff, 97.5):.4f}]')

    # Plot
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(boot_v5, bins=50, alpha=0.6, label='v5 (ไม่ clean)', color='#2c7fb8')
    ax.hist(boot_v6, bins=50, alpha=0.6, label='v6 (clean)', color='#e74c3c')
    ax.set_xlabel('Macro F1')
    ax.set_ylabel('Bootstrap frequency')
    ax.set_title(f'Bootstrap F1 distribution (n={len(agg_v5)} files)\n95% CIs overlap → not statistically significant')
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# === Cell 7: สรุป + ฟังเสียงไฟล์ที่ต่าง (optional) ===
import IPython.display as ipd

if agg_v5 is not None and agg_v6 is not None and len(diff) > 0:
    print('🔊 ฟังเสียงไฟล์ที่ v5/v6 predict ต่างกัน — เช็คเองว่า scam จริงไหม\n')
    for _, row in diff.iterrows():
        fn = row.filename
        print(f"📁 {fn} | true={CLASS_NAMES[row.true]} | "
              f"v5→{CLASS_NAMES[row.pred_v5]} {row.v5_correct} | "
              f"v6→{CLASS_NAMES[row.pred_v6]} {row.v6_correct}")
        path = f'{REAL_AUDIO}/{fn}'
        if os.path.exists(path):
            ipd.display(ipd.Audio(path))
        else:
            print(f'   ⚠️ ไม่พบไฟล์: {path}')
        print()


In [ ]:
# === Cell 8: สรุปข้อความ + คำแนะนำ ===
print(\"\"\"
📊 สรุป v5 vs v6 บน Real Audio
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ผลตัวเลข:
- v5 F1 = ตามที่ report ก่อนหน้า (0.887)
- v6 F1 = ตามที่ report ก่อนหน้า (0.82)
- ต่างกัน 1 ไฟล์ใน 19 = ~5% F1

ข้อจำกัดสำคัญ:
- n=19 เล็กเกิน → variance ใหญ่
- 95% CI ของทั้ง 2 model น่าจะ overlap = ไม่ significant
- Scam recall เท่ากัน (11/12) → จับ scam ดีพอกัน

ข้อเสนอแนะ:
1. รายงาน F1 ทั้ง 2 model + confidence interval (ไม่ใช่ point estimate)
2. ระบุ limitation: real audio น้อย
3. เน้น scam recall (เท่ากัน) มากกว่า F1 (ต่างเล็กน้อย)
4. v6 ดีกว่าที่: methodology, hard negative awareness, deploy quality
\"\"\")
